In [ ]:
!nvidia-smi

Thu Oct 15 11:01:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.9 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
import os
from google.colab import files

os.makedirs("word_surprise", exist_ok=True)

try:
    uploaded = files.upload()
except Exception as error:
    print("The upload did not finish:", error)
    print("Run this cell again, or drag the two files into the Files panel on the left.")

Saving crossentropy_words_spans(3).tsv to crossentropy_words_spans(3).tsv
Saving crossentropy_words_sequences(2).tsv to crossentropy_words_sequences(2).tsv


In [ ]:
import os
import csv
import glob
import json
import bisect
from collections import defaultdict

os.makedirs("word_surprise", exist_ok=True)


def newest(name):
    """The uploaded file called `name`, or its newest copy (`name (1).tsv`)."""
    found = glob.glob(name[:-4] + "*.tsv")
    if not found:
        raise FileNotFoundError(name + " is not here. Upload it in the cell above.")
    path = max(found, key=os.path.getmtime)
    print("reading", path)
    return path


with open(newest("crossentropy_words_sequences.tsv"), encoding="utf-8") as f:
    word_seqs = list(csv.DictReader(f, delimiter="\t"))

spans_by_seq = defaultdict(list)
with open(newest("crossentropy_words_spans.tsv"), encoding="utf-8") as f:
    for row in csv.DictReader(f, delimiter="\t"):
        row["char_start"] = int(row["char_start"])
        row["char_end"] = int(row["char_end"])
        row["word_index"] = int(row["word_index"])
        spans_by_seq[row["seq_id"]].append(row)

print()
print(len(word_seqs), "texts,", sum(len(v) for v in spans_by_seq.values()), "words")

reading crossentropy_words_sequences(2).tsv
reading crossentropy_words_spans(3).tsv

88 texts, 1027 words


In [ ]:
MODELS = {
    "krikri":  "ilsp/Llama-Krikri-8B-Instruct",
    "meltemi": "ilsp/Meltemi-7B-Instruct-v1.5",
    "llama":   "meta-llama/Llama-3.1-8B-Instruct",
}

LOAD_IN_4BIT = True    # same as the other runs

print("4-bit:", LOAD_IN_4BIT)

4-bit: True


In [ ]:
def assign_tokens_to_words(losses, offsets, spans):
    """Give every token to the word it overlaps most.

    losses   loss of each token
    offsets  (start, end) of each token in the text
    spans    the words of the text, sorted by char_start

    Returns (pieces, borrowed). pieces is one list of token losses per word.
    Tokens that touch no word (punctuation, spaces) are left out.
    A word that would get no token at all (a short word inside a piece that
    belongs to its neighbour) borrows that piece, and is marked in borrowed.
    """
    starts = [w["char_start"] for w in spans]
    pieces = [[] for _ in spans]
    nearest = [(0, None) for _ in spans]          # for each word: (overlap, loss) of its best token

    for loss, (t_start, t_end) in zip(losses, offsets):
        if t_end <= t_start:
            continue

        best, best_overlap = None, 0
        i = bisect.bisect_left(starts, t_end) - 1       # last word that starts before the token ends
        while i >= 0 and spans[i]["char_end"] > t_start:
            overlap = min(t_end, spans[i]["char_end"]) - max(t_start, spans[i]["char_start"])
            if overlap > nearest[i][0]:
                nearest[i] = (overlap, loss)
            if overlap > best_overlap:
                best, best_overlap = i, overlap
            i -= 1

        if best is not None:
            pieces[best].append(loss)

    borrowed = [False] * len(spans)
    for i, p in enumerate(pieces):
        if not p and nearest[i][1] is not None:
            pieces[i] = [nearest[i][1]]
            borrowed[i] = True

    return pieces, borrowed


def score_text(seq, spans, losses, offsets, model_name):
    """Turn the token losses of one text into word rows and one text row."""
    per_word, borrowed = assign_tokens_to_words(losses, offsets, spans)

    word_rows = []
    for w, pieces, is_borrowed in zip(spans, per_word, borrowed):
        total = sum(pieces)
        word_rows.append({
            "seq_id": seq["seq_id"], "kind": seq["kind"], "text_id": seq["text_id"],
            "option": seq["option"], "variety": seq["variety"], "model": model_name,
            "word_index": w["word_index"], "word": w["word"], "lemma": w["lemma"],
            "upos": w["upos"], "segment": w["segment"], "sentence_id": w["sentence_id"],
            "shared": w["shared"],
            "n_tokens": len(pieces),
            "borrowed": is_borrowed,
            "loss_sum": round(total, 4),
            "loss_mean": round(total / len(pieces), 4) if pieces else None,
            "token_losses": [round(x, 3) for x in pieces],
        })

    # for an option, only the tokens of the option itself count, not the context before it
    scored_from = int(seq["scored_from"])
    scored = [l for l, (a, b) in zip(losses, offsets) if b > scored_from]

    text_row = {
        "seq_id": seq["seq_id"], "kind": seq["kind"], "text_id": seq["text_id"],
        "option": seq["option"], "variety": seq["variety"], "model": model_name,
        "n_tokens": len(losses), "loss": round(sum(losses) / len(losses), 4),
        "n_tokens_scored": len(scored), "loss_scored": round(sum(scored) / len(scored), 4),
    }
    return word_rows, text_row

In [ ]:
import gc

import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


def load_model(hf_id):
    """Load one model and its tokenizer."""
    tokenizer = AutoTokenizer.from_pretrained(hf_id)

    settings = {"device_map": "auto", "torch_dtype": torch.float16}
    if LOAD_IN_4BIT:
        settings["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
        )

    model = AutoModelForCausalLM.from_pretrained(hf_id, **settings)
    model.eval()
    return model, tokenizer


def token_losses(model, tokenizer, text):
    """Loss for every token, plus where each token sits in the text."""
    encoded = tokenizer(text, return_tensors="pt", return_offsets_mapping=True)
    offsets = encoded.pop("offset_mapping")[0].tolist()
    encoded = {k: v.to(model.device) for k, v in encoded.items()}

    with torch.no_grad():
        outputs = model(**encoded)

    # the model predicts token i+1 from token i, so line them up
    predictions = outputs.logits[0, :-1].float()
    actual = encoded["input_ids"][0, 1:]

    losses = F.cross_entropy(predictions, actual, reduction="none").tolist()

    # losses[i] belongs to the token at offsets[i + 1]
    return losses, offsets[1:]


def run_words(model_name):
    """Score every word of every text and save the results."""
    os.makedirs("word_surprise", exist_ok=True)     # so a restarted session cannot lose a finished run
    print("loading", model_name)
    model, tokenizer = load_model(MODELS[model_name])

    word_rows, text_rows = [], []

    for seq in tqdm(word_seqs, desc=model_name):
        spans = sorted(spans_by_seq[seq["seq_id"]], key=lambda w: w["char_start"])
        losses, offsets = token_losses(model, tokenizer, seq["text"])
        words, text_row = score_text(seq, spans, losses, offsets, model_name)
        word_rows.extend(words)
        text_rows.append(text_row)

    for name, rows in [("words", word_rows), ("seqs", text_rows)]:
        path = "word_surprise/{}_{}.jsonl".format(name, model_name)
        with open(path, "w", encoding="utf-8") as f:
            for row in rows:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
        print("wrote {}  ({} rows)".format(path, len(rows)))

    # free the GPU before the next model
    del model
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
run_words("krikri")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

krikri:   0%|          | 0/88 [00:00<?, ?it/s]

wrote word_surprise/words_krikri.jsonl  (1027 rows)
wrote word_surprise/seqs_krikri.jsonl  (88 rows)
loading krikri


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

krikri:   0%|          | 0/88 [00:00<?, ?it/s]

wrote word_surprise/words_krikri.jsonl  (1027 rows)
wrote word_surprise/seqs_krikri.jsonl  (88 rows)


In [ ]:
run_words("meltemi")

loading meltemi


config.json:   0%|          | 0.00/713 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.65k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.97M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B / 1.18MB            

tokenizer.model: downloading bytes:           |  0.00B            

... (5 outputs left out)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

meltemi:   0%|          | 0/88 [00:00<?, ?it/s]

wrote word_surprise/words_meltemi.jsonl  (1027 rows)
wrote word_surprise/seqs_meltemi.jsonl  (88 rows)


In [ ]:
run_words("llama")

loading llama


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

llama:   0%|          | 0/88 [00:00<?, ?it/s]

wrote word_surprise/words_llama.jsonl  (1027 rows)
wrote word_surprise/seqs_llama.jsonl  (88 rows)


In [ ]:
for model_name in MODELS:
    words_path = "word_surprise/words_{}.jsonl".format(model_name)
    if not os.path.exists(words_path):
        continue

    words = [json.loads(line) for line in open(words_path, encoding="utf-8")]
    texts = [json.loads(line) for line in open("word_surprise/seqs_{}.jsonl".format(model_name), encoding="utf-8")]

    empty = [w for w in words if w["n_tokens"] == 0]
    borrowed = [w for w in words if w["borrowed"]]
    print(model_name, " words =", len(words), " words with 0 pieces =", len(empty),
          " borrowed =", len(borrowed))
    for w in empty[:5]:
        print("     ", w["seq_id"], w["word"])

    for variety in ["pharasiot", "greek"]:
        picked = [w for w in words if w["variety"] == variety and w["n_tokens"] > 0]
        print("   {:10} mean pieces per word {:.2f}   mean word loss {:.2f}".format(
            variety,
            sum(w["n_tokens"] for w in picked) / len(picked),
            sum(w["loss_sum"] for w in picked) / len(picked)))

    earlier = "crossentropy/{}.jsonl".format(model_name)
    if os.path.exists(earlier):
        old = {(r["text_id"], r["variety"]): r["loss"]
               for r in (json.loads(line) for line in open(earlier, encoding="utf-8"))
               if r["level"] == "paragraph"}
        gaps = [abs(t["loss"] - old[(t["text_id"], t["variety"])])
                for t in texts if t["kind"] == "passage" and (t["text_id"], t["variety"]) in old]
        if gaps:
            print("   passage loss against the earlier run: largest difference {:.4f} ({} texts)".format(
                max(gaps), len(gaps)))

    hardest = sorted([w for w in words if w["variety"] == "pharasiot" and w["kind"] == "passage"],
                     key=lambda w: -w["loss_sum"])[:5]
    print("   most surprising Pharasiot words:")
    for w in hardest:
        print("      {:14} {:6.2f}  ({} pieces)  {}".format(w["word"], w["loss_sum"], w["n_tokens"], w["text_id"]))
    print()

krikri  words = 1027  words with 0 pieces = 0  borrowed = 22
   pharasiot  mean pieces per word 2.07   mean word loss 13.39
   greek      mean pieces per word 1.57   mean word loss 6.19
   most surprising Pharasiot words:

meltemi  words = 1027  words with 0 pieces = 0  borrowed = 22
   pharasiot  mean pieces per word 2.04   mean word loss 12.90
   greek      mean pieces per word 1.45   mean word loss 5.51
   most surprising Pharasiot words:

llama  words = 1027  words with 0 pieces = 0  borrowed = 6
   pharasiot  mean pieces per word 2.62   mean word loss 15.33
   greek      mean pieces per word 2.36   mean word loss 7.87
   most surprising Pharasiot words:



In [ ]:
!zip -qr word_surprise_results.zip word_surprise

from google.colab import files

files.download("word_surprise_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>